In [0]:
%sql
use catalog lendingclub;
create schema if not exists gold;
use schema gold;
select current_catalog(), current_schema();

In [0]:
bad_customers_df = spark.sql('select * from delta.`/Volumes/lendingclub/storagelocation/bad_data/Customer/`')
bad_delinq_df = spark.sql('select * from delta.`/Volumes/lendingclub/storagelocation/bad_data/LoanDefaulterDelinq/`')
bad_publicRecord_def = spark.sql('select * from delta.`/Volumes/lendingclub/storagelocation/bad_data/LoanDefaulterPublicRecord/`')

In [0]:
#consolidate all bad data member_id from all datasets and get distinct member_id

bad_customer_data = (bad_customers_df.select('member_id')
                     .union(bad_delinq_df.select('member_id'))
                     .union(bad_publicRecord_def.select('member_id'))
                     )

In [0]:
bad_customer_data.count()

In [0]:
final_bad_customer = bad_customer_data.distinct()

In [0]:
final_bad_customer.count()

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/bad_data/Customer_bad_data_final/", recurse=True)

In [0]:
final_bad_customer.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/bad_data/Customer_bad_data_final/')

In [0]:
%sql
create or replace table bad_data_customer_final
as
select * from delta.`/Volumes/lendingclub/storagelocation/bad_data/Customer_bad_data_final/`

In [0]:
%sql
select * from bad_data_customer_final